<a href="https://colab.research.google.com/github/Rohanrathod7/my-ds-labs/blob/main/17_Mini_project/Customer_Analytics%3A_Preparing_Data_for_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Project Description**  
You've been hired by a major online data science training provider to store their data much more efficiently, so they can create a model that predicts if course enrollees are looking for a job. You'll convert data types, create ordered categories, and filter ordered categorical data so the data is ready for modeling.



The Head Data Scientist at Training Data Ltd. has asked you to create a DataFrame called ds_jobs_transformed that stores the data in customer_train.csv much more efficiently. Specifically, they have set the following requirements:

- Columns containing categories with only two factors must be stored as Booleans (bool).
- Columns containing integers only must be stored as 32-bit integers (int32).
- Columns containing floats must be stored as 16-bit floats (float16).
- Columns containing nominal categorical data must be stored as the category data type.
- Columns containing ordinal categorical data must be stored as ordered categories, and not mapped to numerical values, with an order that reflects the natural order of the column.
- The DataFrame should be filtered to only contain students with 10 or more years of experience at companies with at least 1000 employees, as their recruiter base is suited to more experienced professionals at enterprise companies.

If you call .info() or .memory_usage() methods on ds_jobs and ds_jobs_transformed after you've preprocessed it, you should notice a substantial decrease in memory usage.

In [9]:
# Import necessary libraries
import pandas as pd
import requests
from io import StringIO

# URL of the raw CSV file on GitHub
url = "https://raw.githubusercontent.com/Rohanrathod7/my-ds-labs/main/17_Mini_project/Dataset/customer_train.csv"

# Download the CSV content
response = requests.get(url)
response.raise_for_status() # Raise an exception for bad status codes

# Load the dataset and create a copy
ds_jobs = pd.read_csv(StringIO(response.text))
ds_jobs_transformed = ds_jobs.copy()

# EDA to help identify ordinal, nominal, and two-factor categories
# for col in ds_jobs.select_dtypes("object").columns:
#     print(ds_jobs_transformed[col].value_counts(), '\n')

# Create a dictionary of columns containing ordered categorical data
ordered_cats = {
    'enrolled_university': ['no_enrollment', 'Part time course', 'Full time course'],
    'education_level': ['Primary School', 'High School', 'Graduate', 'Masters', 'Phd'],
    'experience': ['<1'] + list(map(str, range(1, 21))) + ['>20'],
    'company_size': ['<10', '10-49', '50-99', '100-499', '500-999', '1000-4999', '5000-9999', '10000+'],
    'last_new_job': ['never', '1', '2', '3', '4', '>4']
}

# Create a mapping dictionary of columns containing two-factor categories to convert to Booleans
two_factor_cats = {
    'relevant_experience': {'No relevant experience': False, 'Has relevant experience': True},
}

# Loop through DataFrame columns to efficiently change data types
for col in ds_jobs_transformed:

    # Convert two-factor categories to bool
    if col in two_factor_cats.keys():
        ds_jobs_transformed[col] = ds_jobs_transformed[col].map(two_factor_cats[col])

    # Convert 'job_change' to bool
    elif col == 'target':
        ds_jobs_transformed[col] = ds_jobs_transformed[col].astype('bool')

    # Convert integer columns to int32
    elif col in ['enrollee_id', 'training_hours']:
        ds_jobs_transformed[col] = ds_jobs_transformed[col].astype('int32')

    # Convert float columns to float16
    elif col == 'city_development_index':
        ds_jobs_transformed[col] = ds_jobs_transformed[col].astype('float16')

    # Convert columns containing ordered categorical data to ordered categories using dict
    elif col in ordered_cats.keys():
        category = pd.CategoricalDtype(ordered_cats[col], ordered=True)
        ds_jobs_transformed[col] = ds_jobs_transformed[col].astype(category)

    # Convert remaining columns to standard categories
    elif ds_jobs_transformed[col].dtype == 'object':
         ds_jobs_transformed[col] = ds_jobs_transformed[col].astype('category')


# Filter students with 10 or more years experience at companies with at least 1000 employees
ds_jobs_transformed = ds_jobs_transformed[(ds_jobs_transformed['experience'] >= '10') & (ds_jobs_transformed['company_size'] >= '1000-4999')]

# Display memory usage before and after transformation
print("Memory usage before transformation:")
print(ds_jobs.memory_usage(deep=True))
print("\nMemory usage after transformation:")
print(ds_jobs_transformed.memory_usage(deep=True))

# Display info of the transformed DataFrame
print("\nInfo of transformed DataFrame:")
ds_jobs_transformed.info()

Memory usage before transformation:
Index                         132
student_id                 153264
city                      1235888
city_development_index     153264
gender                    1040573
relevant_experience       1527274
enrolled_university       1341257
education_level           1231558
major_discipline          1095945
experience                1121964
company_size              1023519
company_type              1047279
last_new_job              1113264
training_hours             153264
job_change                 153264
dtype: int64

Memory usage after transformation:
Index                     17608
student_id                17608
city                      14289
city_development_index     4402
gender                     2495
relevant_experience        2201
enrolled_university        2525
education_level            2701
major_discipline           2761
experience                 4047
company_size               3008
company_type               2776
last_new_job         